## Required installs

In [149]:
# ==============================================================================
# 0. SETUP AND DATA LOADING (FINAL ABSOLUTE PATH FIX)
# ==============================================================================
library(DiffBind)
library(GenomicRanges)
library(rtracklayer)
library(S4Vectors)
library(tidyverse)
library(data.table)
library(readxl)
# --- Define the correct file paths and directories ---
METADATA_FILE <- "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/atac_seq/ATAC-seq/samples_clean.csv"
CORRECT_BASE_DIR <- "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/atac_seq/ATAC-seq"
diffbind_results_file <- "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/diffbind/dba_all_results.RData"
rna_data_file <- "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/tf_dissection/deseq_temporal_cd133.csv" 
enhancer_atlas_file <- "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/enhancerAtlas_brain_e14.5.txt"
# --- Load TF list ---

tf_file <- "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/Mouse_TFs_Kinases_webpage-3-30-2017.xlsx"
tf_df <- read_excel(tf_file)

# Check structure
colnames(tf_df)

# Get TF symbols (adjust column name based on output above)
tf_list <- unique(tf_df$`Gene Symbol`)  # or tf_df$`Gene Symbol` etc.
# --- A. Load DiffBind results and DARs ---
message(paste("\n[1/4] Loading DiffBind results from:", diffbind_results_file))
load(diffbind_results_file) 

if (!exists("dba_cd133")) {
    stop("Required DiffBind object 'dba_cd133' not found. Cannot proceed.")
}

# 1. Load DARs (E14 vs E18 CD133)
dars_sig <- dba.report(dba_cd133, th = 0.05, fold = 1) 
dar_set_e14 <- dars_sig[dars_sig$Fold > 1] # E14-high
dar_set_e18 <- dars_sig[dars_sig$Fold < -1] # E18-high
message(paste("    - E14-high DARs:", length(dar_set_e14), " E18-high DARs:", length(dar_set_e18)))


# --- B. Create TRUE Consensus PEAKS: Path Manipulation and Loading ---
message("\n[2/4] Creating TRUE CONSENSUS PEAKS using corrected paths (Final attempt)...")

# 2.1 Load metadata and FILTER for the relevant CD133 samples
metadata_df <- readr::read_csv(METADATA_FILE, show_col_types = FALSE)

# Filter for the specific comparison: CD133 (Factor) in E14 or E18 (Condition)
cd133_samples_df <- metadata_df %>%
    dplyr::filter(Factor == "CD133") %>%
    dplyr::filter(Condition %in% c("E14", "E18"))

# 2.2 ISOLATE FILENAMES AND RECONSTRUCT PATHS WITH THE CORRECT DIRECTORY
peak_file_names <- basename(cd133_samples_df$Peaks) # Extract just the 'ISFxxxx_peaks.narrowPeak' part
peak_file_paths_correct <- file.path(CORRECT_BASE_DIR, peak_file_names)

# 2.3 Load and Merge the peaks
all_peaks_list <- list()
for (i in 1:length(peak_file_paths_correct)) {
    df <- tryCatch({
        # narrowPeak files are 0-based start, 1-based end. We load columns 1, 2, 3 (chr, start, end).
        data.table::fread(peak_file_paths_correct[i], header = FALSE, select = c(1:3)) %>% 
             dplyr::rename(seqnames = V1, start = V2, end = V3) %>% 
             dplyr::mutate(start = start + 1) # Convert 0-based start to 1-based GRanges start
    }, error = function(e) {
        message(paste("Error loading file (Path used:", peak_file_paths_correct[i], "):", e$message, "Skipping."))
        return(NULL)
    })
    
    if (!is.null(df)) {
        all_peaks_list[[i]] <- GenomicRanges::makeGRangesFromDataFrame(df, keep.extra.columns = FALSE)
    }
}

# 2.4 Merge and Reduce the list of all peaks to form the True Consensus
all_peaks_union <- unlist(GRangesList(all_peaks_list))
consensus_peaks_true <- GenomicRanges::reduce(all_peaks_union)
consensus_peaks_true <- keepStandardChromosomes(consensus_peaks_true, pruning.mode="coarse")

message(paste("    - Final TRUE CONSENSUS PEAKS (All Accessible Sites):", length(consensus_peaks_true)))


# --- C. Load DESEQ results (for required objects) ---
message(paste("\n[3/4] Loading RNA-seq DE results..."))
rna_df <- readr::read_csv(rna_data_file, show_col_types = FALSE) 
GENE_COL <- "symbol"             
LFC_COL <- "log2FoldChange"     
FDR_COL <- "padj"               
deg_sig <- rna_df %>% dplyr::filter(!!rlang::sym(FDR_COL) < 0.05) %>% dplyr::filter(abs(!!rlang::sym(LFC_COL)) > 1)
deg_list_e14 <- unique(deg_sig %>% dplyr::filter(!!rlang::sym(LFC_COL) > 1) %>% dplyr::pull(!!rlang::sym(GENE_COL)))
deg_list_e18 <- unique(deg_sig %>% dplyr::filter(!!rlang::sym(LFC_COL) < -1) %>% dplyr::pull(!!rlang::sym(GENE_COL)))

[1] "Gene Symbol" "Annotation"  "Family"      "Ensembl ID"  "Uniprot ID"


[1/4] Loading DiffBind results from: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/diffbind/dba_all_results.RData

    - E14-high DARs: 281  E18-high DARs: 426


[2/4] Creating TRUE CONSENSUS PEAKS using corrected paths (Final attempt)...

    - Final TRUE CONSENSUS PEAKS (All Accessible Sites): 93394


[3/4] Loading RNA-seq DE results...

New names:
• `` -> `...1`


In [ ]:
# --- Subset DE genes for TFs ---
tf_de_e14 <- intersect(deg_list_e14, tf_list)
tf_de_e18 <- intersect(deg_list_e18, tf_list)

message(paste("E14-high DE TFs:", length(tf_de_e14)))
print(tf_de_e14)
message(paste("E18-high DE TFs:", length(tf_de_e18)))
print(tf_de_e18)

# --- Check JASPAR overlap ---
jaspar_tf_names <- colData(motif_hits_e14)$name

message("Overlap with JASPAR (case-sensitive):")
print(intersect(tf_de_e14, jaspar_tf_names))
print(intersect(tf_de_e18, jaspar_tf_names))

message("Overlap with JASPAR (case-insensitive):")
print(intersect(toupper(tf_de_e14), toupper(jaspar_tf_names)))
print(intersect(toupper(tf_de_e18), toupper(jaspar_tf_names)))


In [ ]:
# ==============================================================================
# COMBINE ENHANCER ATLAS FILES AND REDO OVERLAP
# ==============================================================================
# --- 1. Load and combine all 3 atlas files ---
message("=== Combining Enhancer Atlas Files ===")

atlas_dir <- "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations"

atlas_files <- c(
    file.path(atlas_dir, "enhancerAtlas_neuron_cortical.txt"),
    file.path(atlas_dir, "enhancerAtlas_brain_e14.5.txt"),
    file.path(atlas_dir, "enhancerAtlas_cortex.txt")
)

# Load and combine
atlas_combined <- rbindlist(lapply(atlas_files, function(f) {
    message(paste("  Loading:", basename(f)))
    fread(f, header = FALSE, sep = "\t", col.names = c("id", "score"))
}))

# Remove duplicates (same enhancer-gene pair)
atlas_combined <- unique(atlas_combined, by = "id")
message(paste("    - Combined entries:", nrow(atlas_combined)))

# --- 2. Parse combined atlas ---
atlas_parsed <- atlas_combined %>%
    tidyr::separate(id, into = c("coords", "gene_info"), sep = "_", extra = "merge") %>%
    tidyr::separate(coords, into = c("chr", "range"), sep = ":") %>%
    tidyr::separate(range, into = c("start", "end"), sep = "-", convert = TRUE) %>%
    tidyr::separate(gene_info, into = c("ensembl", "symbol", "gene_chr", "tss"), 
                    sep = "\\$", fill = "right")

atlas_gr_mm9 <- GenomicRanges::makeGRangesFromDataFrame(
    atlas_parsed,
    keep.extra.columns = TRUE,
    seqnames.field = "chr",
    start.field = "start",
    end.field = "end"
)
atlas_gr_mm9 <- keepStandardChromosomes(atlas_gr_mm9, pruning.mode = "coarse")
message(paste("    - Combined Enhancer Atlas (mm9):", length(atlas_gr_mm9)))
message(paste("    - Unique target genes:", length(unique(atlas_gr_mm9$symbol))))

# --- 3. LiftOver mm9 to mm10 ---
message("\n--- LiftOver mm9 to mm10 ---")
chain_file <- file.path(atlas_dir, "mm9ToMm10.over.chain")
chain <- import.chain(chain_file)

atlas_lifted <- liftOver(atlas_gr_mm9, chain)
atlas_gr <- unlist(atlas_lifted)
message(paste("    - Lifted to mm10:", length(atlas_gr), "of", length(atlas_gr_mm9),
              "(", round(100*length(atlas_gr)/length(atlas_gr_mm9), 1), "%)"))

# --- 4. Overlap DARs with combined Enhancer Atlas ---
message("\n--- Overlapping DARs with Combined Enhancer Atlas (mm10) ---")

hits_e14 <- findOverlaps(dar_set_e14, atlas_gr)
dar_e14_enhancers <- atlas_gr[subjectHits(hits_e14)]
dar_e14_enhancers$dar_idx <- queryHits(hits_e14)

hits_e18 <- findOverlaps(dar_set_e18, atlas_gr)
dar_e18_enhancers <- atlas_gr[subjectHits(hits_e18)]
dar_e18_enhancers$dar_idx <- queryHits(hits_e18)

message(paste("    - E14-high DARs overlapping enhancers:", length(unique(queryHits(hits_e14))), 
              "of", length(dar_set_e14), "(",round(100*length(unique(queryHits(hits_e14)))/length(dar_set_e14),1),"%)"))
message(paste("    - E18-high DARs overlapping enhancers:", length(unique(queryHits(hits_e18))),
              "of", length(dar_set_e18), "(",round(100*length(unique(queryHits(hits_e18)))/length(dar_set_e18),1),"%)"))
message(paste("    - E14 DAR target genes:", length(unique(dar_e14_enhancers$symbol))))
message(paste("    - E18 DAR target genes:", length(unique(dar_e18_enhancers$symbol))))

# --- 5. Overlap Consensus Peaks ---
message("\n--- Overlapping Consensus Peaks with Combined Enhancer Atlas ---")

hits_consensus <- findOverlaps(consensus_peaks_true, atlas_gr)
consensus_enhancers <- atlas_gr[subjectHits(hits_consensus)]
consensus_enhancers$peak_idx <- queryHits(hits_consensus)

message(paste("    - Consensus peaks overlapping enhancers:", length(unique(queryHits(hits_consensus))),
              "of", length(consensus_peaks_true), "(",round(100*length(unique(queryHits(hits_consensus)))/length(consensus_peaks_true),1),"%)"))
message(paste("    - Consensus peak target genes:", length(unique(consensus_enhancers$symbol))))

# --- 6. Summary ---
e14_dar_targets <- unique(dar_e14_enhancers$symbol)
e18_dar_targets <- unique(dar_e18_enhancers$symbol)
consensus_targets <- unique(consensus_enhancers$symbol)

message("\n=== TOP E14 DAR ENHANCER TARGETS ===")
print(head(sort(table(dar_e14_enhancers$symbol), decreasing = TRUE), 20))

message("\n=== TOP E18 DAR ENHANCER TARGETS ===")
print(head(sort(table(dar_e18_enhancers$symbol), decreasing = TRUE), 20))

=== Combining Enhancer Atlas Files ===

  Loading: enhancerAtlas_neuron_cortical.txt

  Loading: enhancerAtlas_brain_e14.5.txt

  Loading: enhancerAtlas_cortex.txt

    - Combined entries: 181331

    - Combined Enhancer Atlas (mm9): 181331

    - Unique target genes: 17897


--- LiftOver mm9 to mm10 ---

    - Lifted to mm10: 181493 of 181331 ( 100.1 %)


--- Overlapping DARs with Combined Enhancer Atlas (mm10) ---

    - E14-high DARs overlapping enhancers: 6 of 281 ( 2.1 %)

    - E18-high DARs overlapping enhancers: 10 of 426 ( 2.3 %)

    - E14 DAR target genes: 25

    - E18 DAR target genes: 42


--- Overlapping Consensus Peaks with Combined Enhancer Atlas ---

    - Consensus peaks overlapping enhancers: 2154 of 93394 ( 2.3 %)

    - Consensus peak target genes: 7012


=== TOP E14 DAR ENHANCER TARGETS ===




         Gin1        Gm3531           Pam       Ppip5k2 2610301G19Rik 
            2             2             2             2             1 
9930012K11Rik B230216N24Rik          Bin3          Egr3        Epb4.9 
            1             1             1             1             1 
       Fam40b       Klhdc10        Klhdc5        Mrps35          Nrf1 
            1             1             1             1             1 
       Pdlim2        Phyhip       Ppfibp1        Ppp3cc        Rad23b 
            1             1             1             1             1 



=== TOP E18 DAR ENHANCER TARGETS ===




      Cttnbp2         Naa38 2310030N02Rik 2610301B20Rik        Aarsd1 
            2             2             1             1             1 
        Adrb3          Aoc2          Aoc3         Becn1          Bmp3 
            1             1             1             1             1 
        Brca1          Brf2        Ccdc56         Dclk1      Eif4ebp1 
            1             1             1             1             1 
       Erlin2          Ezh1       Gm11625       Gm11818        Gpr124 
            1             1             1             1             1 


In [146]:
unique(consensus_enhancers$symbol[(consensus_enhancers$symbol %in% tf_list) & (consensus_enhancers$symbol %in% deg_e14_high$symbol)])

[1] "Lhx9"    "Nhlh1"   "Plagl2"  "Uncx"    "Smad3"   "Lhx1"    "Hmga1"  
 [8] "Bcl11b"  "Neurog1" "Rcor2"

In [147]:
unique(consensus_enhancers$symbol[(consensus_enhancers$symbol %in% tf_list) & (consensus_enhancers$symbol %in% deg_e18_high$symbol)])

[1] "Prrx2"   "Foxs1"   "Bhlhe23" "Foxo1"   "Tfap2e"  "Zfp9"    "Klf2"   
 [8] "Junb"    "Hivep2"  "Gli1"    "Zbtb4"   "Stat5a"  "Mef2c"   "Zfp366" 
[15] "Scx"     "Sox10"   "Csdc2"   "Ppara"   "Dbx2"    "Tbx1"    "Olig1"

In [148]:
# ==============================================================================
# INTERSECT DE GENES WITH CONSENSUS ENHANCER TARGETS
# ==============================================================================

message("\n--- Intersecting DE Genes with Accessible Enhancers ---")

# DE genes with accessible enhancers (consensus peaks)
de_e14_with_enhancers <- intersect(deg_list_e14, consensus_targets)
de_e18_with_enhancers <- intersect(deg_list_e18, consensus_targets)

message(paste("    - E14-high DE genes:", length(deg_list_e14)))
message(paste("    - E14-high DE genes with accessible enhancers:", length(de_e14_with_enhancers),
              "(", round(100*length(de_e14_with_enhancers)/length(deg_list_e14), 1), "%)"))

message(paste("    - E18-high DE genes:", length(deg_list_e18)))
message(paste("    - E18-high DE genes with accessible enhancers:", length(de_e18_with_enhancers),
              "(", round(100*length(de_e18_with_enhancers)/length(deg_list_e18), 1), "%)"))

# --- More stringent: DE genes with DIFFERENTIAL enhancer accessibility (DARs) ---
de_e14_with_e14_dars <- intersect(deg_list_e14, e14_dar_targets)
de_e18_with_e18_dars <- intersect(deg_list_e18, e18_dar_targets)

message("\n--- DE Genes with CONCORDANT Differential Enhancers (DARs) ---")
message(paste("    - E14-high DE genes with E14-high DAR enhancers:", length(de_e14_with_e14_dars),
              "(", round(100*length(de_e14_with_e14_dars)/length(deg_list_e14), 1), "%)"))
message(paste("    - E18-high DE genes with E18-high DAR enhancers:", length(de_e18_with_e18_dars),
              "(", round(100*length(de_e18_with_e18_dars)/length(deg_list_e18), 1), "%)"))

# --- Create summary dataframes ---
# E14 concordant genes (upregulated + more accessible enhancer)
e14_concordant_df <- as.data.frame(dar_e14_enhancers) %>%
    dplyr::filter(symbol %in% deg_list_e14) %>%
    dplyr::select(seqnames, start, end, symbol, score) %>%
    dplyr::distinct() %>%
    dplyr::arrange(symbol)

# E18 concordant genes (upregulated + more accessible enhancer)
e18_concordant_df <- as.data.frame(dar_e18_enhancers) %>%
    dplyr::filter(symbol %in% deg_list_e18) %>%
    dplyr::select(seqnames, start, end, symbol, score) %>%
    dplyr::distinct() %>%
    dplyr::arrange(symbol)

message("\n=== E14 CONCORDANT GENES (DE + DAR enhancer) ===")
print(sort(unique(e14_concordant_df$symbol)))

message("\n=== E18 CONCORDANT GENES (DE + DAR enhancer) ===")
print(sort(unique(e18_concordant_df$symbol)))

# --- Summary table ---
summary_df <- data.frame(
    Category = c("E14-high DE genes", "E18-high DE genes",
                 "E14 DE + any accessible enhancer", "E18 DE + any accessible enhancer",
                 "E14 DE + E14-high DAR enhancer", "E18 DE + E18-high DAR enhancer"),
    Count = c(length(deg_list_e14), length(deg_list_e18),
              length(de_e14_with_enhancers), length(de_e18_with_enhancers),
              length(de_e14_with_e14_dars), length(de_e18_with_e18_dars))
)
print(summary_df)


--- Intersecting DE Genes with Accessible Enhancers ---

    - E14-high DE genes: 255

    - E14-high DE genes with accessible enhancers: 62 ( 24.3 %)

    - E18-high DE genes: 2011

    - E18-high DE genes with accessible enhancers: 507 ( 25.2 %)


--- DE Genes with CONCORDANT Differential Enhancers (DARs) ---

    - E14-high DE genes with E14-high DAR enhancers: 0 ( 0 %)

    - E18-high DE genes with E18-high DAR enhancers: 4 ( 0.2 %)




=== E14 CONCORDANT GENES (DE + DAR enhancer) ===



character(0)



=== E18 CONCORDANT GENES (DE + DAR enhancer) ===



[1] "Adrb3" "Ifi35" "Ptn"   "Ramp2"
                          Category Count
1                E14-high DE genes   255
2                E18-high DE genes  2011
3 E14 DE + any accessible enhancer    62
4 E18 DE + any accessible enhancer   507
5   E14 DE + E14-high DAR enhancer     0
6   E18 DE + E18-high DAR enhancer     4


In [150]:

message("Overlap with Enhancers(case-sensitive):")
print(intersect(tf_de_e14, de_e14_with_enhancers))
print(intersect(tf_de_e18, de_e18_with_enhancers))

Overlap with Enhancers(case-sensitive):



 [1] "Smad3"   "Rcor2"   "Plagl2"  "Lhx9"    "Hmga1"   "Neurog1" "Bcl11b" 
 [8] "Uncx"    "Lhx1"    "Nhlh1"  
 [1] "Csdc2"   "Dbx2"    "Sox10"   "Foxo1"   "Ppara"   "Olig1"   "Zbtb4"  
 [8] "Zfp9"    "Gli1"    "Zfp366"  "Junb"    "Klf2"    "Foxs1"   "Tbx1"   
[15] "Mef2c"   "Stat5a"  "Scx"     "Hivep2"  "Tfap2e"  "Bhlhe23" "Prrx2"  
